In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import seaborn as sns
font_scale = 7
sns.set_theme(style='ticks', font_scale=font_scale, palette=sns.color_palette('Set2'),)
import matplotlib
import matplotlib.pyplot as plt
import polars as pl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datapipe.datasets import NavierStokesGLED

In [115]:
# logs_dir = Path('/home/ttransue/out/ResShift/runs_no_hydra/2025-09-16-20-51')  # 8x: 32 -> 256
# step = 84
# logs_dir = Path('/home/ttransue/out/ResShift/runs_no_hydra/2025-09-16-20-55')  # 16x: 16 -> 256
# step = 90
# logs_dir = Path('/home/ttransue/out/ResShift/runs_no_hydra/2025-09-17-14-32')  # 32x: 8 -> 256
# step = 21
logs_dir = Path('/home/ttransue/out/ResShift/runs_no_hydra/2025-09-17-14-33')  # 64x: 4 -> 256
step = 21
assert logs_dir.exists()
images_dir = logs_dir/'images'/'train'
assert images_dir.exists()

In [116]:
# step = 90
gt = torch.load(images_dir/f'gt-{step}.pt')[0]
lq_gaussian = torch.load(images_dir/f'lq-{step}.pt')[0]
lq_linear = nn.Upsample(size=lq_gaussian.shape[-2:], mode='bilinear')(gt[None])[0]
cfgs = dict(
    x0_GT=gt,
    y0_HQ_linear=nn.Upsample(size=gt.shape[-2:], mode='bilinear')(lq_linear[None])[0],
    y0_HQ_gaussian=nn.Upsample(size=gt.shape[-2:], mode='bilinear')(lq_gaussian[None])[0],
    # xT_latent=torch.load(images_dir/f'diffused-{step}.pt')[0],
    x0_HQ_ResShift=torch.load(images_dir/f'x0-pred-{step}.pt')[0],
    y0_LQ_linear=lq_linear,
    y0_LQ_gaussian=lq_gaussian,
)
# cfgs = {'gt': gt, **{f'gt{i}': _gt['gt'] for i, _gt in enumerate(ds)}}

In [117]:
def get_cmap_bounds(ts, p=.05):
    v_min, v_max = [None, None], [None, None]
    for data in ts:
        for c, channel_data in enumerate(data):
            _v_min = np.quantile(channel_data, p)
            _v_max = np.quantile(channel_data, 1 - p)
            if v_min[c] is None or v_min[c] > _v_min:
                v_min[c] = _v_min
            if v_max[c] is None or v_max[c] > _v_max:
                v_max[c] = _v_max
    return v_min, v_max

In [119]:
show = [
    'trajectory',
    'error',
    'gradient',
][0]
if show == 'trajectory':
    to_plot = {k: v for k, v in cfgs.items()
               if k not in [
                   # 'y0_LQ_linear',
               ]}
elif show == 'error':
    to_plot = {k: v - gt for k, v in cfgs.items()
               if k not in [
                   'y0_HQ_linear',
                   'y0_HQ_gaussian',
                   'y0_LQ_linear',
                   'y0_LQ_gaussian',
               ]}
elif show == 'gradient':
    domain_width, domain_height = 10, 2
    spacing = [domain_width / state.shape[1], domain_height / state.shape[2]]
    to_plot = {k: sum(g.square() for g in torch.gradient(v, spacing=spacing, dim=[1, 2])) / 2#.sqrt()
               for k, v in cfgs.items()
               if k in [
                   'y0_LQ_linear',
                   'y0_LQ_gaussian',
               ]}
    for k in [
                   'y0_LQ_linear',
                   'y0_LQ_gaussian',
    ]:
        to_plot[f'_{k}'] = cfgs[k]
else:
    raise NotImplementedError(f'Unknown show: {show}')
cmap = 'RdBu'
g = (
    sns.FacetGrid(
        data=pd.DataFrame({k: list(map(str, range(gt.shape[0]))) for k in [*to_plot, ':colorbar']}).melt(var_name='Model', value_name='Value'),
        gridspec_kws=dict(
            width_ratios=[*([1] * len(to_plot)), .05],
        ),
        height=20,
        row='Value',
        col='Model',
        sharex=False,
        sharey=False,
    )
    .set_titles(r'$u_{row_name}$|src={col_name}')
)
v_min, v_max = get_cmap_bounds(to_plot.values())
v_absmax = [max(map(abs, (_v_min, _v_max))) for _v_min, _v_max in zip(v_min, v_max)]
for (row, col, hue), _ in g.facet_data():
    ax = g.axes[row, col]
    c = row
    model_name = g.col_names[col]
    if model_name == ':colorbar':
        ax.get_figure().colorbar(matplotlib.cm.ScalarMappable(norm=matplotlib.colors.Normalize(vmin=-v_absmax[c], vmax=v_absmax[c]), cmap=cmap), cax=ax)
        ax.set_title('')
    else:
        data = to_plot[model_name][c]
        if show == 'gradient':
            dirichlet_energy = data
            for dim, s in enumerate(spacing):
                dirichlet_energy = dirichlet_energy.sum(dim=dim, keepdim=True) * s
            ax.set_title(f'{ax.get_title()} ({dirichlet_energy.item():.1f})')
        # ax.set_axis_off()
        im_out = ax.imshow(data, cmap=cmap, vmin=-v_absmax[c], vmax=v_absmax[c])
g.tight_layout()

In [56]:
g.savefig(f'{show}.{step}.pdf', format='pdf', bbox_inches='tight', pad_inches=.03)